In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [10]:
input_path = "../../general_input/"
output_path = "../input/"
train_path = input_path + "train/"
test_path = input_path + "test/"

train_csv = train_path + "train_data_prepared.csv"
embeddings = output_path + "train_embeddings.npy"

train_split_path = output_path + "train/train_fold0_split.csv"
val_split_path = output_path + "val/val_fold0_split.csv"
train_embeddings_split = output_path + "train/train_fold0_embeddings.npy"
val_embeddings_split = output_path + "val/val_fold0_embeddings.npy"

feather_path = "/home/maria/CAFA6-protein-function-prediction/model_650M/input/001_train_proteins_5folds_clust_id25_c80_cov1_mode2_reass1_thr32"

In [11]:
try:
    df = pd.read_feather(feather_path)
    print("Success! Here is the data:")
    print(df.head())
    print(df.columns)
except Exception as e:
    print(f"Could not read as feather: {e}")

Success! Here is the data:
   protein_id                                           sequence  seq_length  \
0  A0A0C5B5G6                                   MRWQEMGYIFYPRKLR          16   
1      A0JNW5  MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...        1464   
2      A0JP26  MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...         581   
3      A0PK11  MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...         232   
4      A1A4S6  MGLQPLEFSDCYLDSPWFRERIRAHEAELERTNKFIKELIKDGKNL...         786   

   taxon      rep_id  fold  
0   9606  A0A0C5B5G6     2  
1   9606      A0JNW5     3  
2   9606      Q6S8J3     3  
3   9606  A0A2R8RY99     2  
4   9606      A1A4S6     2  
Index(['protein_id', 'sequence', 'seq_length', 'taxon', 'rep_id', 'fold'], dtype='object')


In [21]:
df.columns = ['id', 'sequence', 'seq_length', 'taxon', 'rep_id', 'fold']

In [13]:
df_complete = pd.read_csv(train_csv)
embeddings_complete = np.load(embeddings)

In [24]:
df_combined = pd.merge(df_complete, df, on='id', how='left')
df_combined.head()

,db,id,name,full_name,organism,taxon_x,gen,PE,SV,sequence_x,length,go_terms,aspect,sequence_y,seq_length,taxon_y,rep_id,fold
0,sp,A0A023FBW4,E1142_AMBCJ,Evasin P1142,Amblyomma cajennense,34607,NaN,3,1,MTSHGAVKIAIFAVIALHSIFECLSKPQILQRTDHSTDSDWDPQMC...,97,[['GO:0019958']],['F'],MTSHGAVKIAIFAVIALHSIFECLSKPQILQRTDHSTDSDWDPQMC...,97,34607,A0A023FBW4,4
1,sp,A0A023FBW7,EV546_AMBCJ,Evasin P546,Amblyomma cajennense,34607,NaN,3,1,MKVLLYIAASCLMLLALNVSAENTQQEEEDYDYGTDTCPFPVLANK...,118,[['GO:0019957']],['F'],MKVLLYIAASCLMLLALNVSAENTQQEEEDYDYGTDTCPFPVLANK...,118,34607,A0A023FBW7,1
2,sp,A0A023FDY8,EV974_AMBCJ,Evasin P974,Amblyomma cajennense,34607,NaN,1,1,MKVLLCIAASCLMLLALNVSAENTQQEEQDYDYGTDTCPFPVLANK...,118,[['GO:0019957']],['F'],MKVLLCIAASCLMLLALNVSAENTQQEEQDYDYGTDTCPFPVLANK...,118,34607,A0A023FBW7,1
3,sp,A0A023FF81,E1126_AMBCJ,Evasin P1126,Amblyomma cajennense,34607,NaN,3,1,MTSHSAVRIAIFAVIALHSIFECLSKPQILQRTDKSTDSEWDPQTC...,90,[['GO:0019958']],['F'],MTSHSAVRIAIFAVIALHSIFECLSKPQILQRTDKSTDSEWDPQTC...,90,34607,A0A023FBW4,4
4,sp,A0A023FFB5,EV983_AMBCJ,Evasin P983,Amblyomma cajennense,34607,NaN,3,2,MKASFCVIASCLVVFALKGTAEDTGTEDDFDYGNTGCPFPVLGNYK...,119,[['GO:0019957']],['F'],MKASFCVIASCLVVFALKGTAEDTGTEDDFDYGNTGCPFPVLGNYK...,119,34607,L7M8Z8,2


In [25]:
train_df = df_combined[df_combined['fold'] != 0].copy().reset_index(drop=True)
val_df = df_combined[df_combined['fold'] == 0].copy().reset_index(drop=True)

print(f"Train shape: {train_df.shape} and Val shape: {val_df.shape}")

Train shape: (65754, 18) and Val shape: (16650, 18)


In [27]:
mask_val = (df_combined['fold'] == 0).values
mask_train = (df_combined['fold'] != 0).values

In [28]:
print(f"Original embeddings shape: {embeddings_complete.shape}")

Original embeddings shape: (82404, 1280)


In [29]:
X_val = embeddings_complete[mask_val]
X_train = embeddings_complete[mask_train]

print(f"Train embeddings shape: {X_train.shape}")
print(f"Val embeddings shape: {X_val.shape}")

Train embeddings shape: (65754, 1280)
Val embeddings shape: (16650, 1280)


In [30]:
# Save splits
np.save(train_embeddings_split, X_train)
np.save(val_embeddings_split, X_val)

train_df.to_csv(train_split_path, index=False)
val_df.to_csv(val_split_path, index=False)